In [1]:
import time
import numpy as np
import pandas as pd



In [2]:
TRAIN_PATH = "../input/ventilator-pressure-prediction/train.csv"
TEST_PATH = "../input/ventilator-pressure-prediction/test.csv"
SAMPLE_SUB_PATH = "../input/ventilator-pressure-prediction/sample_submission.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
df_sub = pd.read_csv(SAMPLE_SUB_PATH)

assert "id" in df_sub.columns and "pressure" in df_sub.columns
assert df_sub[
    "id"
].is_monotonic_increasing, "Expected sample_submission ids to be sorted."
assert df_sub["id"].nunique() == len(df_sub), "Submission id must be unique."
assert set(df_sub["id"].values) == set(
    df_test["id"].values
), "Submission ids must match test ids."



In [3]:
unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)



In [4]:
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else upper_val
    )


def snap_to_nearest_pressure_vec(pred_arr: np.ndarray) -> np.ndarray:
    idx = np.searchsorted(sorted_pressures, pred_arr, side="left")
    idx0 = np.clip(idx, 0, total_pressures_len - 1)
    idxm1 = np.clip(idx - 1, 0, total_pressures_len - 1)

    upper = sorted_pressures[idx0]
    lower = sorted_pressures[idxm1]

    choose_lower = np.abs(lower - pred_arr) < np.abs(upper - pred_arr)
    out = np.where(choose_lower, lower, upper)

    out = np.where(idx == 0, sorted_pressures[0], out)
    out = np.where(idx == total_pressures_len, sorted_pressures[-1], out)
    return out




In [5]:
t0 = time.time()


def add_breath_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["breath_id", "time_step"], kind="mergesort").copy()
    g = df.groupby("breath_id", sort=False)

    df["u_in_lag1"] = g["u_in"].shift(1).fillna(0.0)
    df["u_in_delta1"] = df["u_in"] - df["u_in_lag1"]
    df["u_in_cumsum"] = g["u_in"].cumsum()
    df["u_in_x_time"] = df["u_in"] * df["time_step"]

    df["t_idx"] = g.cumcount().astype(np.int16)
    return df


df_train_fe = add_breath_features(df_train)
df_test_fe = add_breath_features(df_test)

train_all = df_train_fe
train_insp = df_train_fe[df_train_fe["u_out"] == 0].copy()

# Build design matrices (same features / same model family as before).
u_in_tr_all = train_all["u_in"].to_numpy(dtype=np.float64)
u_out_tr_all = train_all["u_out"].to_numpy(dtype=np.float64)
R_tr_all = train_all["R"].to_numpy(dtype=np.float64)
C_tr_all = train_all["C"].to_numpy(dtype=np.float64)
ts_tr_all = train_all["time_step"].to_numpy(dtype=np.float64)

u_in_lag1_tr_all = train_all["u_in_lag1"].to_numpy(dtype=np.float64)
u_in_delta1_tr_all = train_all["u_in_delta1"].to_numpy(dtype=np.float64)
u_in_cumsum_tr_all = train_all["u_in_cumsum"].to_numpy(dtype=np.float64)
u_in_x_time_tr_all = train_all["u_in_x_time"].to_numpy(dtype=np.float64)

X_all = np.c_[
    np.ones(len(train_all), dtype=np.float64),
    u_in_tr_all,
    u_out_tr_all,
    R_tr_all,
    C_tr_all,
    u_in_tr_all * R_tr_all,
    u_in_tr_all * C_tr_all,
    ts_tr_all,
    u_in_lag1_tr_all,
    u_in_delta1_tr_all,
    u_in_cumsum_tr_all,
    u_in_x_time_tr_all,
]
y_all = train_all["pressure"].to_numpy(dtype=np.float64)

# Keep your original global fit (used for completeness / stability on all rows).
beta_all, *_ = np.linalg.lstsq(X_all, y_all, rcond=None)

# CHANGE (score improvement toward target):
# Fit an inspiratory-only beta using the same least-squares logic and same features.
# This better matches the metric-scored regime and makes residuals more consistent.
u_in_tr = train_insp["u_in"].to_numpy(dtype=np.float64)
R_tr = train_insp["R"].to_numpy(dtype=np.float64)
C_tr = train_insp["C"].to_numpy(dtype=np.float64)
ts_tr = train_insp["time_step"].to_numpy(dtype=np.float64)
u_in_lag1_tr = train_insp["u_in_lag1"].to_numpy(dtype=np.float64)
u_in_delta1_tr = train_insp["u_in_delta1"].to_numpy(dtype=np.float64)
u_in_cumsum_tr = train_insp["u_in_cumsum"].to_numpy(dtype=np.float64)
u_in_x_time_tr = train_insp["u_in_x_time"].to_numpy(dtype=np.float64)
u_out_tr = train_insp["u_out"].to_numpy(dtype=np.float64)

X_insp = np.c_[
    np.ones(len(train_insp), dtype=np.float64),
    u_in_tr,
    u_out_tr,
    R_tr,
    C_tr,
    u_in_tr * R_tr,
    u_in_tr * C_tr,
    ts_tr,
    u_in_lag1_tr,
    u_in_delta1_tr,
    u_in_cumsum_tr,
    u_in_x_time_tr,
]
y_insp = train_insp["pressure"].to_numpy(dtype=np.float64)
beta_insp, *_ = np.linalg.lstsq(X_insp, y_insp, rcond=None)

# Predict on test (use insp-fit only on insp rows; keep your exp-row baseline logic unchanged below).
u_in_te = df_test_fe["u_in"].to_numpy(dtype=np.float64)
u_out_te_raw = df_test_fe["u_out"].to_numpy(dtype=np.float64)
R_te = df_test_fe["R"].to_numpy(dtype=np.float64)
C_te = df_test_fe["C"].to_numpy(dtype=np.float64)
ts_te = df_test_fe["time_step"].to_numpy(dtype=np.float64)

u_in_lag1_te = df_test_fe["u_in_lag1"].to_numpy(dtype=np.float64)
u_in_delta1_te = df_test_fe["u_in_delta1"].to_numpy(dtype=np.float64)
u_in_cumsum_te = df_test_fe["u_in_cumsum"].to_numpy(dtype=np.float64)
u_in_x_time_te = df_test_fe["u_in_x_time"].to_numpy(dtype=np.float64)

X_test = np.c_[
    np.ones(len(df_test_fe), dtype=np.float64),
    u_in_te,
    u_out_te_raw,
    R_te,
    C_te,
    u_in_te * R_te,
    u_in_te * C_te,
    ts_te,
    u_in_lag1_te,
    u_in_delta1_te,
    u_in_cumsum_te,
    u_in_x_time_te,
]

# Base predictions: inspiratory uses beta_insp (metric-aligned), expiratory uses beta_all (won't matter after baseline override).
pred = X_test @ beta_all
insp_mask = u_out_te_raw == 0.0
pred = np.where(insp_mask, (X_test @ beta_insp), pred)

# Residuals computed consistently with inspiratory-only beta.
train_pred_insp = X_insp @ beta_insp
train_resid = y_insp - train_pred_insp

UIN_BIN_SIZE = 1.0
UIN_LAG_BIN_SIZE = 2.0

u_in_bin_tr = np.floor(u_in_tr / UIN_BIN_SIZE).astype(np.int16)
u_in_bin_te = np.floor(u_in_te / UIN_BIN_SIZE).astype(np.int16)

u_in_lag1_bin_tr = np.floor(u_in_lag1_tr / UIN_LAG_BIN_SIZE).astype(np.int16)
u_in_lag1_bin_te = np.floor(u_in_lag1_te / UIN_LAG_BIN_SIZE).astype(np.int16)

UIN_CUMSUM_BIN_SIZE = 25.0  # coarse to limit table size/runtime
u_in_cumsum_bin_tr = np.floor(u_in_cumsum_tr / UIN_CUMSUM_BIN_SIZE).astype(np.int16)
u_in_cumsum_bin_te = np.floor(u_in_cumsum_te / UIN_CUMSUM_BIN_SIZE).astype(np.int16)

TS_BIN_SIZE = 0.03  # ~1 bin per timestep (time_step increments are ~0.03)
ts_bin_tr = np.floor(ts_tr / TS_BIN_SIZE).astype(np.int16)
ts_bin_te = np.floor(ts_te / TS_BIN_SIZE).astype(np.int16)

ts_r_tr = np.round(ts_tr, 2)
ts_r_te = np.round(ts_te, 2)
u_in_r_tr = np.round(u_in_tr, 1)
u_in_r_te = np.round(u_in_te, 1)
u_in_lag1_r_tr = np.round(u_in_lag1_tr, 1)
u_in_lag1_r_te = np.round(u_in_lag1_te, 1)
u_in_cumsum_r_tr = np.round(u_in_cumsum_tr, 0)
u_in_cumsum_r_te = np.round(u_in_cumsum_te, 0)

resid_table_rc_t_ts_u_ul_uc_round = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(
        ts_r=ts_r_tr,
        u_in_r=u_in_r_tr,
        u_in_lag1_r=u_in_lag1_r_tr,
        u_in_cumsum_r=u_in_cumsum_r_tr,
        resid=train_resid,
    )
    .groupby(
        ["R", "C", "t_idx", "ts_r", "u_in_r", "u_in_lag1_r", "u_in_cumsum_r"],
        sort=False,
    )["resid"]
    .mean()
)

resid_table_rc_t_u_ul_uc_ts = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(
        ts_bin=ts_bin_tr,
        u_in_bin=u_in_bin_tr,
        u_in_lag1_bin=u_in_lag1_bin_tr,
        u_in_cumsum_bin=u_in_cumsum_bin_tr,
        resid=train_resid,
    )
    .groupby(
        ["R", "C", "t_idx", "ts_bin", "u_in_bin", "u_in_lag1_bin", "u_in_cumsum_bin"],
        sort=False,
    )["resid"]
    .mean()
)

resid_table_rc_t_u_ul_uc = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(
        u_in_bin=u_in_bin_tr,
        u_in_lag1_bin=u_in_lag1_bin_tr,
        u_in_cumsum_bin=u_in_cumsum_bin_tr,
        resid=train_resid,
    )
    .groupby(
        ["R", "C", "t_idx", "u_in_bin", "u_in_lag1_bin", "u_in_cumsum_bin"], sort=False
    )["resid"]
    .mean()
)

resid_table_rc_t_u_ul = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(u_in_bin=u_in_bin_tr, u_in_lag1_bin=u_in_lag1_bin_tr, resid=train_resid)
    .groupby(["R", "C", "t_idx", "u_in_bin", "u_in_lag1_bin"], sort=False)["resid"]
    .mean()
)

resid_table_rc_t_u = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(u_in_bin=u_in_bin_tr, resid=train_resid)
    .groupby(["R", "C", "t_idx", "u_in_bin"], sort=False)["resid"]
    .mean()
)

resid_table_rc_t = (
    train_insp.loc[:, ["R", "C", "t_idx"]]
    .assign(resid=train_resid)
    .groupby(["R", "C", "t_idx"], sort=False)["resid"]
    .mean()
)

R_te_k = df_test_fe["R"].to_numpy()
C_te_k = df_test_fe["C"].to_numpy()
t_idx_te_k = df_test_fe["t_idx"].to_numpy()

test_keys_round = list(
    zip(
        R_te_k, C_te_k, t_idx_te_k, ts_r_te, u_in_r_te, u_in_lag1_r_te, u_in_cumsum_r_te
    )
)
corr_round = np.fromiter(
    (resid_table_rc_t_ts_u_ul_uc_round.get(k, np.nan) for k in test_keys_round),
    dtype=np.float64,
    count=len(test_keys_round),
)

test_keys_u_ul_uc_ts = list(
    zip(
        R_te_k,
        C_te_k,
        t_idx_te_k,
        ts_bin_te,
        u_in_bin_te,
        u_in_lag1_bin_te,
        u_in_cumsum_bin_te,
    )
)
corr_u_ul_uc_ts = np.fromiter(
    (resid_table_rc_t_u_ul_uc_ts.get(k, np.nan) for k in test_keys_u_ul_uc_ts),
    dtype=np.float64,
    count=len(test_keys_u_ul_uc_ts),
)

test_keys_u_ul_uc = list(
    zip(R_te_k, C_te_k, t_idx_te_k, u_in_bin_te, u_in_lag1_bin_te, u_in_cumsum_bin_te)
)
corr_u_ul_uc = np.fromiter(
    (resid_table_rc_t_u_ul_uc.get(k, np.nan) for k in test_keys_u_ul_uc),
    dtype=np.float64,
    count=len(test_keys_u_ul_uc),
)

test_keys_u_ul = list(zip(R_te_k, C_te_k, t_idx_te_k, u_in_bin_te, u_in_lag1_bin_te))
corr_u_ul = np.fromiter(
    (resid_table_rc_t_u_ul.get(k, np.nan) for k in test_keys_u_ul),
    dtype=np.float64,
    count=len(test_keys_u_ul),
)

test_keys_u = list(zip(R_te_k, C_te_k, t_idx_te_k, u_in_bin_te))
corr_u = np.fromiter(
    (resid_table_rc_t_u.get(k, np.nan) for k in test_keys_u),
    dtype=np.float64,
    count=len(test_keys_u),
)

test_keys_rc_t = list(zip(R_te_k, C_te_k, t_idx_te_k))
corr_rc_t = np.fromiter(
    (resid_table_rc_t.get(k, 0.0) for k in test_keys_rc_t),
    dtype=np.float64,
    count=len(test_keys_rc_t),
)

corr = np.where(
    np.isnan(corr_round),
    np.where(
        np.isnan(corr_u_ul_uc_ts),
        np.where(
            np.isnan(corr_u_ul_uc),
            np.where(
                np.isnan(corr_u_ul),
                np.where(np.isnan(corr_u), corr_rc_t, corr_u),
                corr_u_ul,
            ),
            corr_u_ul_uc,
        ),
        corr_u_ul_uc_ts,
    ),
    corr_round,
)

# Apply residual correction only for inspiratory rows (as before).
pred = np.where(insp_mask, pred + corr, pred)

mean_insp_table = (
    train_insp.loc[:, ["R", "C", "t_idx", "pressure"]]
    .groupby(["R", "C", "t_idx"], sort=False)["pressure"]
    .mean()
)
global_insp_mean = float(train_insp["pressure"].mean())

baseline = np.fromiter(
    (mean_insp_table.get(k, global_insp_mean) for k in test_keys_rc_t),
    dtype=np.float64,
    count=len(test_keys_rc_t),
)

exp_mask = u_out_te_raw == 1.0
pred = np.where(exp_mask, baseline, pred)

pred = np.clip(pred, sorted_pressures[0], sorted_pressures[-1])

pred_df = pd.DataFrame({"id": df_test_fe["id"].to_numpy(), "pressure": pred})
pred_df["pressure"] = snap_to_nearest_pressure_vec(
    pred_df["pressure"].to_numpy(dtype=np.float64)
)

df_sub = df_sub[["id"]].merge(pred_df, on="id", how="left", validate="one_to_one")

elapsed = time.time() - t0
print(
    f"Done in {elapsed:.2f}s. beta_all shape={beta_all.shape}, beta_insp shape={beta_insp.shape}, "
    f"resid_table_round size={len(resid_table_rc_t_ts_u_ul_uc_round)}, "
    f"resid_table_rc_t_u_ul_uc_ts size={len(resid_table_rc_t_u_ul_uc_ts)}, "
    f"resid_table_rc_t_u_ul_uc size={len(resid_table_rc_t_u_ul_uc)}, "
    f"resid_table_rc_t_u_ul size={len(resid_table_rc_t_u_ul)}, "
    f"resid_table_rc_t_u size={len(resid_table_rc_t_u)}, resid_table_rc_t size={len(resid_table_rc_t)}"
)



Done in 55.11s. beta_all shape=(12,), beta_insp shape=(12,), resid_table_round size=1783490, resid_table_rc_t_u_ul_uc_ts size=521369, resid_table_rc_t_u_ul_uc size=454095, resid_table_rc_t_u_ul size=151196, resid_table_rc_t_u size=20575, resid_table_rc_t size=288


In [6]:
assert list(df_sub.columns) == [
    "id",
    "pressure",
], "Submission must have exactly ['id','pressure'] columns."
assert len(df_sub) == len(df_test), "Submission row count must match test row count."
assert df_sub["pressure"].notna().all(), "Predicted pressures must not be NaN."
assert df_sub[
    "id"
].is_monotonic_increasing, (
    "Submission ids must be sorted increasing like sample_submission."
)
assert df_sub["id"].nunique() == len(df_sub), "Submission id must be unique."
assert (
    df_sub["id"].values == df_sub["id"].sort_values().values
).all(), "Submission ids must be sorted."

df_sub.head()



,id,pressure
0,1,5.626585
1,2,5.837492
2,3,6.540513
3,4,8.157463
4,5,11.250757


In [7]:
OUT_PATH = "submission.csv"
df_sub.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} with shape={df_sub.shape}")

Wrote submission.csv with shape=(603600, 2)
